# Music source separation with a U-Net (a Computer Vision approach)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alessandro1040/audio-separation-unet/blob/main/notebooks/separate_audio_with_unet.ipynb)

Upload **any audio file** and separate it into **vocals / drums / bass / other** with
the U-Net trained in this repository, then see *in detail what the model did*:

```
audio ──STFT──► spectrogram image ──U-Net──► 4 masks ──mask ⊙ mixture──► iSTFT ──► 4 wavs
                                  (segmentation)     (cut the sources out of the photo)
```

You get: the 4 stems (playable/downloadable), the mixture spectrogram with the predicted
masks, the magnitude spectrograms of the estimates, and a report with levels, mask
statistics and - if you also upload ground-truth stems - SI-SDR and BSS-Eval
(SDR/SIR/SAR). See `models/MODEL_CARD.md`: it is a deliberately small model trained on a
laptop, it beats the trivial baseline, it is not state of the art.

Python 3 + CPU is enough (a 3-minute song takes ~1 minute); *Runtime → GPU* is faster.

## 1. Install dependencies and clone the repository (the weights are inside it)

In [ ]:
!pip -q install soundfile mir_eval matplotlib numpy scipy pyyaml 2>&1 | tail -1
%cd /content
!rm -rf audio-separation-unet     # always take the current commit, never a stale clone
!git clone --depth 1 https://github.com/Alessandro1040/audio-separation-unet.git
%cd audio-separation-unet
!ls -la models/
import subprocess, torch
print('repo commit:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
      capture_output=True, text=True).stdout.strip())
ckpt = torch.load('models/unet_musdb18_ema.pt', map_location='cpu', weights_only=False)
print('checkpoint keys:', list(ckpt))
print('trained for', ckpt['step'], 'steps; validation SI-SDR',
      round(ckpt['validation_si_sdr_db'], 2), 'dB on 14 held-out MUSDB18 songs')
print('weights:', ckpt.get('weights'), ckpt.get('dtype'))
if ckpt['step'] < 2250:
    print('*** WARNING: this clone predates the retrained checkpoint. Go back one cell,'
          ' add  !rm -rf audio-separation-unet  and re-run the clone. ***')

In [ ]:
# self-test of the pipeline shipped in the repo (STFT round-trip, masks, U-Net, metrics)
!python -m pytest tests/ -q -p no:warnings 2>&1 | tail -3

## 2. Choose the audio to separate

Upload your own file (any format: wav / mp3 / flac / m4a / ogg, any length)...

In [ ]:
from google.colab import files

uploaded = files.upload()
AUDIO_PATH = next(iter(uploaded))
print('using:', AUDIO_PATH)

In [ ]:
# ...or, if you skipped the upload, use the example mixture shipped in the repository
AUDIO_PATH = 'samples/Al James - Schoolboy Facination - mixture (input).mp3'

from IPython.display import Audio, display
display(Audio(AUDIO_PATH))

## 3. Run the separator (this is the whole app)

`src/analyze.py` does everything: STFT, sliding-window U-Net inference, masks, iSTFT,
figures and the report.

In [ ]:
import os, shutil

OUT = 'outputs/colab_demo'
shutil.rmtree(OUT, ignore_errors=True)
!python -m src.analyze --checkpoint models/unet_musdb18_ema.pt \
    --input "$AUDIO_PATH" --out $OUT --figure-seconds 20

print('\nfiles produced:')
for name in sorted(os.listdir(OUT)):
    print(' ', name, round(os.path.getsize(os.path.join(OUT, name)) / 1e6, 2), 'MB')

### Sanity check - is it separating, or only re-levelling?

Two numbers tell the two failure modes apart, without trusting your ears. `mean sum of
masks` should be ~1.0 (a pure gain re-levelling gives ~0.74), and the four stems should be
*different* signals: the step-1200 checkpoint produced stems that correlated 0.88 with each
other (worst pair 0.98, i.e. the same audio at another level), the retrained one sits at
~0.6.

In [ ]:
import itertools, json
import numpy as np, soundfile as sf

stems = {s: sf.read(f'{OUT}/{s}.wav', always_2d=True)[0]
         for s in ('vocals', 'drums', 'bass', 'other')}
sim = [float(np.dot(stems[a].ravel(), stems[b].ravel()) /
             (np.linalg.norm(stems[a]) * np.linalg.norm(stems[b]) + 1e-12))
       for a, b in itertools.combinations(stems, 2)]
bands = json.load(open(f'{OUT}/report.json'))['mask_statistics']
print('mean sum of masks  :', round(bands['mean_sum_of_masks'], 2),
      '(~1.0 calibrated, ~0.74 = re-levelling)')
print('stem similarity    :', round(float(np.mean(sim)), 2),
      '(~0.6 separating, ~0.88 = the old checkpoint)')
bass = bands['mean_abs_mask_per_frequency_band']['bass']
print('bass mask low/high :', round(bass['low_<200Hz'], 2), '/',
      round(bass['high_>2kHz'], 2), '(low must win)')
print('checkpoint step    :', ckpt['step'],
      '- a 1200 here means you are still running the old weights')

### Step 1-2 — what the network sees, and the masks it predicted

Each mask is a per-source transparency template: ~1 where that instrument is present,
~0 where it is not. The U-Net predicts all four jointly, for every time-frequency pixel.

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'{OUT}/masks.png'))

### Step 3 — mask ⊙ mixture spectrum → iSTFT → audio

In [ ]:
display(Image(filename=f'{OUT}/stems.png'))

In [ ]:
for stem in ['vocals', 'drums', 'bass', 'other']:
    print(stem)
    display(Audio(f'{OUT}/{stem}.wav'))

print('mixture (what the model was given), for an A/B comparison')
display(Audio(f'{OUT}/mixture.wav'))

## 4. The report: the numbers behind the separation

* **mixture consistency** — the four estimates are projected so they sum back to the
  input: a residual near −140 dB means the DSP pipeline is exact (a free self-check).
* **mean sum of masks** — ~1.0 would be a perfectly calibrated 4-source mask set.
* **mean |mask| per band** — where each mask is open on average: a *bass* mask should be
  heavier below 200 Hz than above 2 kHz. It is the mean **per bin**, not a share of the
  mask mass: with ~9 bins below 200 Hz and ~930 above 2 kHz, a share would mostly count
  bins. This shows what the model learned (and, at this training budget, what it still
  has to learn).
* **centroid est vs ref** — with ground truth available, a large mismatch means the
  estimated stem still contains other instruments.

In [ ]:
print(open(f'{OUT}/report.txt').read())

In [ ]:
# optional: SI-SDR and BSS-Eval against ground truth - upload vocals.wav, drums.wav,
# bass.wav, other.wav into /content/stems and uncomment:
#
# !python -m src.analyze --checkpoint models/unet_musdb18_ema.pt --input "$AUDIO_PATH" \
#     --out outputs/with_reference --reference-dir /content/stems --bss-seconds 30

## 5. Download the stems

The four wav files are in `outputs/colab_demo/`; zip them and download, or use the Colab
file browser (folder icon on the left).

In [ ]:
!zip -q -j stems.zip $OUT/vocals.wav $OUT/drums.wav $OUT/bass.wav $OUT/other.wav
files.download('stems.zip')

---
### Honest notes and limitations

* Trained on **MUSDB18** (100 songs, 4 stems) for ~1 hour on a laptop (Apple M5):
  10.5 M parameters, 2500 steps. It beats the trivial "mixture as every source"
  baseline (+3.6 dB SI-SDR over 10 MUSDB test songs) but it is far from Demucs-class
  systems (~9 dB BSS-Eval SDR), which are waveform models trained on much more compute.
* The first version of these weights could not separate at all - it only re-levelled the
  mixture. The cause was a bug in the training augmentation (a stem was swapped with a
  *random* stem of another song, so three training targets out of four carried the wrong
  instrument). It is fixed; the README section *Why the first trained model only changed
  the volumes* has the measurement, and `figures/masks_before_after.png` shows the masks
  before and after.
* Works best on **music** with those four stems; speech/podcasts are out of domain.
* Vocals are the hardest source (they overlap with everything in time-frequency);
  bass and "other" come out best.
* Everything runs locally in this Colab: no API, no hidden pre-processing. The code is
  in `src/dsp.py` (STFT/iSTFT), `src/models/unet.py` (the network),
  `src/models/separation.py` (masks → audio) and `src/analyze.py` (this app).
* Want a better model? The README explains how to resume training
  (`python -m src.train --resume checkpoints/last.pt ...`).